In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# See what files we have
files = os.listdir("../data/raw/amazon_catalog/")
print(f"Total files: {len(files)}")
print("CSV files:", [f for f in files if f.endswith(".csv")][:5], "...")

# Load main catalog
df = pd.read_csv("../data/raw/amazon_catalog/Amazon-Products.csv", on_bad_lines="skip")
print(f"\nMain catalog shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")


Total files: 141
CSV files: ['Air Conditioners.csv', 'All Appliances.csv', 'All Books.csv', 'All Car and Motorbike Products.csv', 'All Electronics.csv'] ...

Main catalog shape: (551585, 10)
Columns: ['Unnamed: 0', 'name', 'main_category', 'sub_category', 'image', 'link', 'ratings', 'no_of_ratings', 'discount_price', 'actual_price']


In [2]:
print(df.head(3))
print("\nDtypes:\n", df.dtypes)


   Unnamed: 0                                               name  \
0           0  Lloyd 1.5 Ton 3 Star Inverter Split Ac (5 In 1...   
1           1  LG 1.5 Ton 5 Star AI DUAL Inverter Split AC (C...   
2           2  LG 1 Ton 4 Star Ai Dual Inverter Split Ac (Cop...   

  main_category      sub_category  \
0    appliances  Air Conditioners   
1    appliances  Air Conditioners   
2    appliances  Air Conditioners   

                                               image  \
0  https://m.media-amazon.com/images/I/31UISB90sY...   
1  https://m.media-amazon.com/images/I/51JFb7FctD...   
2  https://m.media-amazon.com/images/I/51JFb7FctD...   

                                                link ratings no_of_ratings  \
0  https://www.amazon.in/Lloyd-Inverter-Convertib...     4.2         2,255   
1  https://www.amazon.in/LG-Convertible-Anti-Viru...     4.2         2,948   
2  https://www.amazon.in/LG-Inverter-Convertible-...     4.2         1,206   

  discount_price actual_price  
0       

In [3]:
print("Missing values:")
print(df.isnull().sum())
print("\nMissing %:")
print((df.isnull().sum() / len(df) * 100).round(2))


Missing values:
Unnamed: 0             0
name                   0
main_category          0
sub_category           0
image                  0
link                   0
ratings           175794
no_of_ratings     175794
discount_price     61163
actual_price       17813
dtype: int64

Missing %:
Unnamed: 0         0.00
name               0.00
main_category      0.00
sub_category       0.00
image              0.00
link               0.00
ratings           31.87
no_of_ratings     31.87
discount_price    11.09
actual_price       3.23
dtype: float64


In [4]:
print("Unique categories:", df.iloc[:, 0].nunique() if df.shape[1] > 0 else "N/A")

# Find price column
price_cols = [c for c in df.columns if "price" in c.lower() or "rate" in c.lower()]
print("Price-related columns:", price_cols)

for col in price_cols:
    print(f"\n{col}:")
    print(df[col].describe())


Unique categories: 19200
Price-related columns: ['discount_price', 'actual_price']

discount_price:
count     490422
unique     27511
top         ₹499
freq       18248
Name: discount_price, dtype: object

actual_price:
count     533772
unique     23170
top         ₹999
freq       48774
Name: actual_price, dtype: object


In [5]:
cat_cols = [c for c in df.columns if "categ" in c.lower() or "dept" in c.lower()]
print("Category columns:", cat_cols)

for col in cat_cols[:2]:
    print(f"\nTop 10 {col}:")
    print(df[col].value_counts().head(10))


Category columns: ['main_category', 'sub_category']

Top 10 main_category:
main_category
accessories            116141
men's clothing          76656
women's clothing        76512
tv, audio & cameras     68659
men's shoes             57456
appliances              33096
stores                  32903
home & kitchen          14568
kids' fashion           13488
sports & fitness        12648
Name: count, dtype: int64

Top 10 sub_category:
sub_category
Formal Shoes      19200
Sports Shoes      19200
Men's Fashion     19200
Western Wear      19200
Shirts            19200
Jeans             19200
Clothing          19152
Bags & Luggage    19152
Jewellery         19152
Innerwear         19152
Name: count, dtype: int64


In [7]:
# Find name/title and description columns
text_cols = [c for c in df.columns if any(k in c.lower() for k in ["name","title","desc","about"])]
print("Text columns:", text_cols)

for col in text_cols[:3]:
    non_null = df[col].dropna()
    print(f"\n{col}:")
    print(f"  Non-null: {len(non_null):,}")
    print(f"  Avg length: {non_null.astype(str).str.len().mean():.0f} chars")
    print(f"  Sample: {str(non_null.iloc[0])[:100]}")



Text columns: ['Unnamed: 0', 'name']

Unnamed: 0:
  Non-null: 551,585
  Avg length: 4 chars
  Sample: 0

name:
  Non-null: 551,585
  Avg length: 79 chars
  Sample: Lloyd 1.5 Ton 3 Star Inverter Split Ac (5 In 1 Convertible, Copper, Anti-Viral + Pm 2.5 Filter, 2023


In [8]:
# Preview price cleaning
sample = df["actual_price"].dropna().head(5)
print("Raw prices:", sample.values)
cleaned = sample.str.replace("₹", "", regex=False).str.replace(",", "", regex=False).astype(float)
print("Cleaned prices:", cleaned.values)

print("""
AMAZON CATALOG EDA SUMMARY
===========================
- 551,585 products across 10 main categories
- Top categories: accessories, men's clothing, women's clothing, electronics
- NO description column — only product name (79 chars avg)
- Prices stored as strings with ₹ symbol and commas → need parsing
- 31.87% missing ratings — keep, still usable for enrichment
- image and link columns → not needed, drop

CLEANING PLAN:
- Drop: Unnamed: 0, image, link
- Clean discount_price, actual_price → strip ₹ and commas → float
- Clean no_of_ratings → strip commas → int
- Convert ratings → float
- Drop rows where name is null (none currently)
- Claude Haiku will generate descriptions from name + category
- Save as clean catalog for LLM enrichment pipeline
""")



Raw prices: ['₹58,990' '₹75,990' '₹61,990' '₹68,990' '₹67,790']
Cleaned prices: [58990. 75990. 61990. 68990. 67790.]

AMAZON CATALOG EDA SUMMARY
- 551,585 products across 10 main categories
- Top categories: accessories, men's clothing, women's clothing, electronics
- NO description column — only product name (79 chars avg)
- Prices stored as strings with ₹ symbol and commas → need parsing
- 31.87% missing ratings — keep, still usable for enrichment
- image and link columns → not needed, drop

CLEANING PLAN:
- Drop: Unnamed: 0, image, link
- Clean discount_price, actual_price → strip ₹ and commas → float
- Clean no_of_ratings → strip commas → int
- Convert ratings → float
- Drop rows where name is null (none currently)
- Claude Haiku will generate descriptions from name + category
- Save as clean catalog for LLM enrichment pipeline

